# MedViLL — Modernized
**Medical Vision-Language Learning** | Retrieval · Classification · VQA · Report Generation

---

### Credits
- **Original paper**: Moon et al., *"Multi-modal Understanding and Generation for Medical Images and Text via Vision-Language Pre-Training"*, IEEE JBHI 2022. [GitHub](https://github.com/SuperSupermoon/MedViLL)
- **OpenI dataset**: Demner-Fushman et al., *"Preparing a collection of radiology examinations for distribution and retrieval"*, JAMIA 2016. DOI: [10.1093/jamia/ocv080](https://doi.org/10.1093/jamia/ocv080)
- **MIMIC-CXR**: Johnson et al., Scientific Data 2019. DOI: [10.1038/s41597-019-0322-0](https://doi.org/10.1038/s41597-019-0322-0)
- **VQA-RAD**: Lau et al., Scientific Data 2018. DOI: [10.1038/sdata.2018.251](https://doi.org/10.1038/sdata.2018.251)

---
**Runtime**: GPU (Runtime → Change runtime type → T4 GPU)  
**Repo**: https://github.com/schizocatto/medvill_fix

## 0 — Environment check

In [ ]:
import torch, platform, subprocess

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"GPU     : {gpu}")
print(f"RAM     : {subprocess.getoutput('free -h | grep Mem').split()[1]}"
      if platform.system() == "Linux" else "")

## 1 — Clone repo & install dependencies

In [ ]:
import os

REPO_URL  = "https://github.com/schizocatto/medvill_fix.git"
REPO_DIR  = "/content/medvill_fix"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
%%capture install_log
# Colab already has torch; we only need the extra deps.
# --quiet keeps the cell output clean; check install_log if anything fails.
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.29.0 \
    einops>=0.7.0 \
    sacrebleu>=2.3.1 \
    rouge-score>=0.1.2 \
    omegaconf>=2.3.0 \
    evaluate>=0.4.1 \
    fuzzywuzzy>=0.18.0 \
    python-Levenshtein>=0.25.0

print("✓ Dependencies installed")

In [ ]:
import sys, importlib

# Add repo root to path so `import medvill` works
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Smoke-test core imports
import medvill
from medvill.config import MedViLLConfig, ModelConfig, ImageConfig, TrainingConfig
from medvill.models import (
    MedViLL,
    MedViLLForClassification,
    MedViLLForRetrieval,
    MedViLLForVQA,
    MedViLLForGeneration,
)
from medvill.metrics import (
    compute_bleu4, RunningPerplexity,
    compute_retrieval_metrics,
    compute_classification_metrics,
)
from medvill.utils import set_seed, get_logger

print(f"✓ medvill {medvill.__version__} imported successfully")
print(f"  device: {'cuda' if __import__('torch').cuda.is_available() else 'cpu'}")

## 2 — Mount Google Drive & point to your data

Your JSONL files and pre-training checkpoints live in Drive.  
Adjust the paths below to match your folder layout.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ── Edit these paths to match your Drive layout ──────────────────────────────
DRIVE_ROOT       = "/content/drive/MyDrive/MedViLL"

PRETRAIN_CKPT    = f"{DRIVE_ROOT}/checkpoints/pretrain_best.pt"   # or None

# Classification (MIMIC-CXR / OpenI JSONL)
CLS_TRAIN        = f"{DRIVE_ROOT}/data/cls_train.jsonl"
CLS_VAL          = f"{DRIVE_ROOT}/data/cls_val.jsonl"
CLS_TEST         = f"{DRIVE_ROOT}/data/cls_test.jsonl"

# Retrieval (same JSONL format: {img, text})
RET_TRAIN        = f"{DRIVE_ROOT}/data/retrieval_train.jsonl"
RET_VAL          = f"{DRIVE_ROOT}/data/retrieval_val.jsonl"

# VQA-RAD directory (trainset.json / testset.json / images/ / cache/)
VQA_DIR          = f"{DRIVE_ROOT}/data/vqa_rad"

# Report generation (same JSONL format)
GEN_TRAIN        = f"{DRIVE_ROOT}/data/gen_train.jsonl"
GEN_VAL          = f"{DRIVE_ROOT}/data/gen_val.jsonl"

OUTPUT_DIR       = f"{DRIVE_ROOT}/outputs"

print("Drive paths configured.")

## 3 — Shared config & tokenizer

One config object is shared across all experiments; override per-task fields inline.

In [ ]:
import torch
from omegaconf import OmegaConf
from transformers import AutoTokenizer
from medvill.config import MedViLLConfig, ModelConfig, ImageConfig, DataConfig, TrainingConfig
from medvill.utils import set_seed

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── Base config (tweak any field here) ───────────────────────────────────────
cfg = MedViLLConfig(
    model=ModelConfig(bert_model="bert-base-uncased"),
    image=ImageConfig(
        encoder_type="resnet50",
        img_size=512,
        num_image_embeds=180,
        img_hidden_sz=2048,
    ),
    data=DataConfig(
        train_path="PLACEHOLDER",   # overridden per task
        seq_len=128,
        num_workers=2,              # 2 is safe on Colab
    ),
    training=TrainingConfig(
        output_dir=OUTPUT_DIR,
        epochs=10,
        batch_size=16,              # reduce to 8 if OOM
        lr=5e-5,
        fp16=True,
        seed=42,
        gradient_accumulation_steps=2,
    ),
)

TOKENIZER = AutoTokenizer.from_pretrained(cfg.model.bert_model)
print(f"Tokenizer vocab size: {TOKENIZER.vocab_size}")

## 4 — Experiment A: Diagnosis Classification

Multi-label classification (14 MIMIC-CXR labels or OpenI labels).  
Metrics: macro AUC-ROC, macro F1.

In [ ]:
import json
from torch.utils.data import DataLoader
from medvill.data import ClassificationDataset
from medvill.models import MedViLLForClassification
from medvill.tasks import ClassificationTrainer
from medvill.utils import load_pretrained_encoder

# ── Build label map from training file ───────────────────────────────────────
def build_label_map(jsonl_path):
    labels = set()
    with open(jsonl_path) as f:
        for line in f:
            raw = json.loads(line).get("label", "")
            if isinstance(raw, str):
                labels.update(l.strip() for l in raw.split(",") if l.strip())
    return {l: i for i, l in enumerate(sorted(labels))}

label2idx   = build_label_map(CLS_TRAIN)
label_names = [k for k, v in sorted(label2idx.items(), key=lambda x: x[1])]
print(f"Labels ({len(label2idx)}): {label_names}")

# ── Datasets & loaders ───────────────────────────────────────────────────────
cls_cfg = cfg   # re-use shared config
cls_cfg.data.train_path = CLS_TRAIN
cls_cfg.data.val_path   = CLS_VAL

train_ds = ClassificationDataset(CLS_TRAIN, TOKENIZER, label2idx,
                                  seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                  train=True)
val_ds   = ClassificationDataset(CLS_VAL,   TOKENIZER, label2idx,
                                  seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                  train=False)

train_loader = DataLoader(train_ds, batch_size=cfg.training.batch_size,
                          shuffle=True,  num_workers=cfg.data.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.training.batch_size,
                          shuffle=False, num_workers=cfg.data.num_workers)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# ── Model ─────────────────────────────────────────────────────────────────────
cls_model = MedViLLForClassification(cfg.model, cfg.image,
                                      num_labels=len(label2idx), multilabel=True)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    missing, unexpected = load_pretrained_encoder(cls_model, PRETRAIN_CKPT)
    print(f"Loaded pretrained encoder | missing={len(missing)} unexpected={len(unexpected)}")

# ── Train ─────────────────────────────────────────────────────────────────────
cls_trainer = ClassificationTrainer(cls_model, train_loader, val_loader,
                                     cfg, DEVICE, label_names)
cls_trainer.train()

## 5 — Experiment B: Retrieval (I2T and T2I)

Cross-encoder retrieval using the ITM head.  
Metrics: Recall@1/5/10, MRR, median rank — computed for both I2T and T2I directions.

In [ ]:
from medvill.data import RetrievalDataset, RetrievalEvalDataset
from medvill.models import MedViLLForRetrieval
from medvill.tasks import RetrievalTrainer

ret_train_ds = RetrievalDataset(RET_TRAIN, TOKENIZER,
                                 seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                 train=True)
ret_eval_ds  = RetrievalEvalDataset(RET_VAL, TOKENIZER,
                                     seq_len=cfg.data.seq_len, img_size=cfg.image.img_size)

ret_train_loader = DataLoader(ret_train_ds, batch_size=cfg.training.batch_size,
                               shuffle=True, num_workers=cfg.data.num_workers, pin_memory=True)

print(f"Train: {len(ret_train_ds)} | Eval: {len(ret_eval_ds)}")

ret_model = MedViLLForRetrieval(cfg.model, cfg.image)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    load_pretrained_encoder(ret_model, PRETRAIN_CKPT)
    print("Loaded pretrained encoder")

ret_trainer = RetrievalTrainer(ret_model, ret_train_loader, ret_eval_ds, cfg, DEVICE)
ret_trainer.train()

## 6 — Experiment C: VQA on VQA-RAD

Visual Question Answering on radiology images.  
Set `organ` to `"chest"`, `"head"`, `"abd"`, or `None` (all).  
Metrics: Accuracy + BLEU-4 on answer strings.

In [ ]:
from medvill.data import VQARadDataset
from medvill.models import MedViLLForVQA
from medvill.tasks import VQATrainer
from medvill.metrics import compute_bleu4

VQA_ORGAN = None   # "chest" | "head" | "abd" | None

vqa_train_ds = VQARadDataset(VQA_DIR, TOKENIZER, split="train", organ=VQA_ORGAN,
                              seq_len=cfg.data.seq_len, img_size=cfg.image.img_size, train=True)
vqa_test_ds  = VQARadDataset(VQA_DIR, TOKENIZER, split="test",  organ=VQA_ORGAN,
                              seq_len=cfg.data.seq_len, img_size=cfg.image.img_size, train=False)

# Share vocab built from train set
vqa_test_ds.ans2label = vqa_train_ds.ans2label
vqa_test_ds.label2ans = vqa_train_ds.label2ans
vqa_test_ds.num_answers = vqa_train_ds.num_answers

print(f"Train: {len(vqa_train_ds)} | Test: {len(vqa_test_ds)}")
print(f"Answer vocab size: {vqa_train_ds.num_answers}")

vqa_train_loader = DataLoader(vqa_train_ds, batch_size=cfg.training.batch_size,
                               shuffle=True,  num_workers=cfg.data.num_workers, pin_memory=True)
vqa_test_loader  = DataLoader(vqa_test_ds,  batch_size=cfg.training.batch_size,
                               shuffle=False, num_workers=cfg.data.num_workers)

vqa_model = MedViLLForVQA(cfg.model, cfg.image, num_answers=vqa_train_ds.num_answers)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    load_pretrained_encoder(vqa_model, PRETRAIN_CKPT)
    print("Loaded pretrained encoder")

vqa_trainer = VQATrainer(vqa_model, vqa_train_loader, vqa_test_loader,
                          cfg, DEVICE, label2ans=vqa_train_ds.label2ans)
vqa_trainer.train()

# ── Post-training metrics ─────────────────────────────────────────────────────
print("\n── Test-set metrics ──")
pred_texts, gt_texts = vqa_trainer.predict(vqa_test_loader)
bleu = compute_bleu4(pred_texts, gt_texts)
print(f"BLEU-4 : {bleu['bleu4']:.2f}")
print(f"BLEU-1 : {bleu['bleu1']:.2f}")
print(f"BLEU-2 : {bleu['bleu2']:.2f}")
print(f"BLEU-3 : {bleu['bleu3']:.2f}")

## 7 — Experiment D: Report Generation

Seq2seq radiology report generation.  
Metrics: BLEU-4 (sacrebleu) and Perplexity.

In [ ]:
from medvill.data import ReportGenerationDataset
from medvill.models import MedViLLForGeneration
from medvill.tasks import GenerationTrainer

gen_train_ds = ReportGenerationDataset(GEN_TRAIN, TOKENIZER,
                                        seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                        train=True)
gen_val_ds   = ReportGenerationDataset(GEN_VAL,   TOKENIZER,
                                        seq_len=cfg.data.seq_len, img_size=cfg.image.img_size,
                                        train=False)

gen_train_loader = DataLoader(gen_train_ds, batch_size=cfg.training.batch_size,
                               shuffle=True,  num_workers=cfg.data.num_workers, pin_memory=True)
gen_val_loader   = DataLoader(gen_val_ds,   batch_size=cfg.training.batch_size,
                               shuffle=False, num_workers=cfg.data.num_workers)

print(f"Train: {len(gen_train_ds)} | Val: {len(gen_val_ds)}")

gen_model = MedViLLForGeneration(cfg.model, cfg.image)
if PRETRAIN_CKPT and os.path.exists(PRETRAIN_CKPT):
    load_pretrained_encoder(gen_model, PRETRAIN_CKPT)
    print("Loaded pretrained encoder")

gen_trainer = GenerationTrainer(gen_model, gen_train_loader, gen_val_loader,
                                 TOKENIZER, cfg, DEVICE)
gen_trainer.train()

## 8 — Standalone evaluation on a saved checkpoint

Use this cell to evaluate any saved checkpoint without retraining.

In [ ]:
EVAL_TASK       = "vqa"        # classification | retrieval | vqa | generation
EVAL_CHECKPOINT = f"{OUTPUT_DIR}/vqa/vqa_best_epoch009_step0.pt"   # adjust path

import subprocess, sys

cmd = [
    sys.executable, f"{REPO_DIR}/scripts/evaluate.py",
    "--task",       EVAL_TASK,
    "--config",     f"{REPO_DIR}/configs/{EVAL_TASK}.yaml",
    "--checkpoint", EVAL_CHECKPOINT,
]

# Task-specific path args
if EVAL_TASK == "vqa":
    cmd += ["--vqa_dir", VQA_DIR]
elif EVAL_TASK == "classification":
    cmd += ["--data_path", CLS_TEST]
elif EVAL_TASK == "retrieval":
    cmd += ["--data_path", RET_VAL]
elif EVAL_TASK == "generation":
    cmd += ["--data_path", GEN_VAL]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])